---

<h1 align="center"><b>REGRESIÓN LINEAL RIDGE Y LASSO</b></h1> 

<p style="text-align: justify; line-height: 1.5; font-size: 16px;">  
La regresión Ridge y LASSO son extensiones de la regresión lineal diseñadas para mejorar el desempeño del modelo cuando existen problemas como la multicolinealidad o el sobreajuste. Ambas incorporan un término de regularización que penaliza la magnitud de los coeficientes, permitiendo obtener modelos más estables y con mejor capacidad de generalización. Mientras Ridge reduce los coeficientes sin eliminarlos, LASSO puede llevar algunos a cero, realizando de forma implícita una selección de variables. Estas técnicas son ampliamente utilizadas en ciencia de datos porque equilibran precisión predictiva e interpretabilidad del modelo.

</p>

---

## Regresión RIDGE (L2)
    
Definición: Agrega penalización cuadrática a los coeficientes.
    
$$ \min \left[ \sum(y_i - \hat{y}_i)^2 + \lambda \sum \beta_j^2 \right] $$
    
**Intuición**
    
- Reduce magnitud de coeficientes

- No elimina variables
    
**Uso**
 
- Multicolinealidad

- Modelos más estables
   
---

## Regresión LASSO (L1)
    
Definición : Penaliza con valor absoluto.
    
$$ \min \left[ \sum(y_i - \hat{y}_i)^2 + \lambda \sum |\beta_j| \right] $$
  
 **Intuición**

- Puede hacer coeficientes exactamente 0

- Selección automática de variables
  
**Uso**

- Reducir dimensionalidad

- Interpretabilidad
 
 ---

<div align="center">

| Modelo | Penalización | Selección de variables |
| :--- | :--- | :--- |
| Ridge | $L2$ | No |
| LASSO | $L1$ | Sí |

</div>

---

In [ ]:
# ===================== #
# 1. CARGA DE LIBRERÍAS #
# ===================== #

import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, Ridge, Lasso

In [ ]:
# =================== #
# 2. LECTURA DE RUTAS #
# =================== #

mainpath= "/Users/duvancatano/Documents/Data_Analytics_UdeA/ml-project/data/housing"
filename= "housing_dataset.csv"
fullpath= os.path.join(mainpath,filename)


In [ ]:
# ================= #
# 3. CARGA DE DATOS #
# ================= #

df = pd.read_csv(fullpath, sep=",") # El separador es "," porque en el archivo .csv los valores están separados por coma
pd.set_option('display.max_columns', None) # Para mostrar todas las columnas del DataFrame sin truncar

In [ ]:
# ========================= #
# 4. SELECCIÓN DE VARIABLES #
# ========================= #

cat_cols = [
    'municipio',
    'zona_urbana_rural',
    'tipo_de_vivienda',
    'acceso_a_transporte_publico',
    'acceso_a_internet',
    'acceso_a_agua_potable',
    'acceso_a_gas'
]

num_cols = [
    'area_total_m2',  
    'ingreso_del_hogar',
    'distancia_al_centro_de_la_ciudad_km'
]

int_cols = [
    'habitaciones', 
    'banos',
    'antiguedad_de_la_vivienda', 
    'parqueaderos',
    'estrato',
]

target = 'valor_de_mercado_de_la_vivienda'

X = df[cat_cols + num_cols + int_cols]
y = df[target]

In [ ]:
# ================ #
# 5. PREPROCESADOR #
# ================ #

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols),
    ('num', StandardScaler(), num_cols + int_cols)
]
                                 )

In [ ]:
# =============================== #
# VISUALIZACIÓN DEL PREPROCESADOR #
# =============================== #

preprocessor

In [ ]:
# =========================== #
# 6. MODELOS EN CONSIDERACIÓN #
# =========================== #

models = {
    'Linear': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.1)
}

In [ ]:
# ===================== #
# 7. VALIDACIÓN CRUZADA #
# ===================== #

results = {}

for name, model in models.items():
    pipe = Pipeline([
        ('preprocessing', preprocessor),
        ('model', model)
    ])

    scores = cross_validate(
        pipe, X, y, cv=5,
        scoring={
            'mae': 'neg_mean_absolute_error',
            'rmse': 'neg_root_mean_squared_error',
            'r2': 'r2'
        }
    )

    results[name] = {
        'MAE': -scores['test_mae'].mean(),
        'RMSE': -scores['test_rmse'].mean(),
        'R2': scores['test_r2'].mean()
    }

pd.DataFrame(results).T

In [ ]:
# ====================== #
# COMPARACIÓN DE MODELOS #
# ====================== #

import matplotlib.pyplot as plt

results_df = pd.DataFrame(results).T

results_df.plot(kind='bar')
plt.title("Comparación de Modelos")
plt.show()

In [ ]:
# ======================== #
# IMPORTANCIA DE VARIABLES #
# ======================== #

pipe.fit(X, y)

coef = pipe.named_steps['model'].coef_
names = pipe.named_steps['preprocessing'].get_feature_names_out()

importance = pd.Series(coef, index=names).sort_values(key=abs, ascending=False)
print(importance.head(10))

In [ ]:
# =========================================== #
# GRÁFICA DEL TOP DE IMPORTANCIA DE VARIABLES #
# =========================================== #

top_importance = importance.head(10)

plt.figure()

top_importance.sort_values().plot(kind='barh')

plt.title("Top 10 Variables Más Importantes")
plt.xlabel("Coeficiente")
plt.ylabel("Variable")

plt.show()

---

<h1 align="center"><b>Ejercicio</b></h1> 

---

<p style="text-align: justify; line-height: 1.5; font-size: 16px;">  

### **Selección de $\lambda$ en Ridge y LASSO :**

Implementar un procedimiento de búsqueda de hiperparámetros para los modelos de Regresión Ridge y LASSO, con el objetivo de identificar el valor óptimo de $\lambda.$

---

$$
\hat{\beta}^{Ridge}(\lambda) = \arg\min_{\beta} \left\{ 
\sum_{i=1}^{n} (y_i - X_i \beta)^2 + \lambda \sum_{j=1}^{p} \beta_j^2 
\right\},

\ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \

\hat{\beta}^{LASSO}(\lambda) = \arg\min_{\beta} \left\{ 
\sum_{i=1}^{n} (y_i - X_i \beta)^2 + \lambda \sum_{j=1}^{p} |\beta_j| 
\right\}

$$

---

**Instrucciones**

1. Defina una grilla de valores de $\alpha$ (por ejemplo, en escala logarítmica entre $10^{-4}$ y $10^{2}$



2. Para cada valor de $\alpha:$

- Ajuste modelos Ridge y LASSO.

- Evalúe su desempeño mediante validación cruzada (k-fold).

- Calcule el RMSE promedio.



3. Organice los resultados en una tabla y determine:

- El mejor modelo Ridge.

- El mejor modelo LASSO.

- El modelo final (menor RMSE).



4. Análisis

- Grafique RMSE vs $\alpha$ para ambos modelos.

- Interprete el efecto de la regularización.

</p>


---

# 🎬 **¡FIN!**

---

<div style="text-align: justify; line-height: 1.5; font-size: 18px;">    </div>